In [103]:
# Système
import os
import sys
import time

# Built-in imports
import warnings
from collections import Counter
import pickle

# Data manipulation
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import missingno

# Sklearn imports
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    root_mean_squared_error,
    mean_absolute_error
)

# ML Models - Linear
from sklearn.linear_model import (
    LogisticRegression,
    Perceptron,
    SGDClassifier,
    Lasso,
    LassoCV
)


from sklearn.preprocessing import LabelEncoder

# ML Models - SVM
from sklearn.svm import SVC, LinearSVC

# ML Models - Ensemble
from sklearn.ensemble import RandomForestClassifier

# ML Models - Other
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# Disable warnings
warnings.filterwarnings('ignore')

In [104]:
from google.colab import drive
drive.mount("/content/drive/", force_remount=True)

Mounted at /content/drive/


In [105]:
def process_dataframes_flexible(df, mappings=None, columns_to_drop=None):
    """
    Version flexible permettant de personnaliser les mappings et colonnes à supprimer

    Args:
        df : DataFrame original
        mappings : dict des mappings personnalisés
        columns_to_drop : dict des colonnes à supprimer personnalisées

    Returns:
        tuple : (df_admi_non_admi, df_session, df_mention)
    """

    # Mappings par défaut
    default_mappings = {
        'resultat': {'NON ADMIS': 0, 'AUTORISE': 1, 'PASSE': 1},
        'session': {'Deuxième Session': 0, 'Première Session': 1},
        'mention': {'Passable': 0, 'Assez-Bien': 1, 'Bien': 1, 'Très-Bien': 1}
    }

    # Colonnes à supprimer par défaut
    default_columns_to_drop = {
        'admi_non_admi': ['COME', 'AFTA', 'EPAT', 'SVT', 'CREDIT', 'SESSION', 'MENTION',
                         'Résultat', 'Type candidature', 'Rang DAP', 'Score DAP'],
        'session': ['COME', 'AFTA', 'EPAT', 'CREDIT', 'SVT', 'MENTION', 'RESULTAT',
                   'Résultat', 'Type candidature', 'Rang DAP', 'Score DAP'],
        'mention': ['COME', 'AFTA', 'EPAT', 'CREDIT', 'SVT', 'SESSION', 'RESULTAT',
                   'Résultat', 'Type candidature', 'Rang DAP', 'Score DAP']
    }

    # Utiliser les paramètres fournis ou les valeurs par défaut
    mappings = mappings or default_mappings
    columns_to_drop = columns_to_drop or default_columns_to_drop

    # Traitement identique à la fonction précédente
    df_admi_non_admi = df.copy()
    df_admi_non_admi['RESULTAT'] = df_admi_non_admi['RESULTAT'].map(mappings['resultat'])
    df_admi_non_admi = df_admi_non_admi.drop(columns_to_drop['admi_non_admi'], axis=1)

    df_session = df.copy()
    df_session['RESULTAT'] = df_session['RESULTAT'].map(mappings['resultat'])
    df_session = df_session.loc[df_session['RESULTAT'] == 1].copy()
    df_session['SESSION'] = df_session['SESSION'].map(mappings['session'])
    df_session = df_session.drop(columns_to_drop['session'], axis=1)

    df_mention = df.copy()
    df_mention['RESULTAT'] = df_mention['RESULTAT'].map(mappings['resultat'])
    df_mention = df_mention.loc[df_mention['RESULTAT'] == 1].copy()
    df_mention['SESSION'] = df_mention['SESSION'].map(mappings['session'])
    df_mention = df_mention.loc[df_mention['SESSION'] == 1].copy()
    df_mention['MENTION'] = df_mention['MENTION'].map(mappings['mention'])
    df_mention = df_mention.drop(columns_to_drop['mention'], axis=1)


    df_admi_non_admi = df_admi_non_admi.drop(['MOYENNE ANNUELLE', 'Mention'],axis=1)
    df_session = df_session.drop(['MOYENNE ANNUELLE', 'Mention'],axis=1)
    df_mention = df_mention.drop(['MOYENNE ANNUELLE', 'Mention'],axis=1)

    return df_admi_non_admi, df_session, df_mention

In [106]:
def split_train_test_flexible(dataframes_config, test_size=0.2, random_state=42):
    """
    Version flexible pour gérer différentes configurations de DataFrames

    Args:
        dataframes_config : dict avec la configuration pour chaque DataFrame
        Format: {
            'nom_cas': {
                'df': DataFrame,
                'target': 'nom_colonne_cible',
                'drop_columns': ['col1', 'col2', ...]
            }
        }
        test_size : proportion des données de test
        random_state : graine aléatoire

    Returns:
        dict : dictionnaire contenant tous les ensembles train/test
    """
    from sklearn.model_selection import train_test_split

    results = {}

    for case_name, config in dataframes_config.items():
        df = config['df']
        target_col = config['target']
        drop_columns = config['drop_columns'] + [target_col]

        # Préparer X et y
        X = df.drop(drop_columns, axis=1)
        y = df[target_col]

        # Split train/test
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )

        # Stocker les résultats
        results[f'X_train_{case_name}'] = X_train
        results[f'X_test_{case_name}'] = X_test
        results[f'y_train_{case_name}'] = y_train
        results[f'y_test_{case_name}'] = y_test
        results[f'X_{case_name}'] = X
        results[f'y_{case_name}'] = y

    return results

In [107]:
def train_evaluate_models(X_train, y_train, X_test):
    """
    Entraîne et évalue plusieurs modèles de classification.

    Args:
        X_train: Features d'entraînement
        y_train: Labels d'entraînement
        X_test: Features de test

    Returns:
        dict: Dictionnaire contenant les scores d'accuracy pour chaque modèle
    """

    models = {
        'SVC': SVC(),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'Gaussian': GaussianNB(),
        'LinearSVC': LinearSVC(),
        'SGD': SGDClassifier(),
        'DecisionTree': DecisionTreeClassifier(),
        'RandomForest': RandomForestClassifier(n_estimators=100),
        'XGBoost': XGBClassifier(learning_rate=0.0001)
    }

    results = {}

    for name, model in models.items():
        # Entraînement du modèle
        model.fit(X_train, y_train)

        # Prédictions
        y_pred = model.predict(X_test)

        # Calcul du score
        accuracy = round(model.score(X_train, y_train) * 100, 2)
        results[name] = accuracy

    return results

In [108]:
models = None

def train_evaluate_models_flexible(X_train, y_train, X_test, df_type, exclude_from_save=None):
    """
    Entraîne et évalue plusieurs modèles, avec possibilité d'exclure certains de la sauvegarde

    Args:
        X_train, y_train, X_test : données d'entraînement et de test
        df_type : type de DataFrame pour nommer le fichier sauvegardé
        exclude_from_save : liste des noms de modèles à exclure de la sauvegarde

    Returns:
        dict : résultats d'accuracy pour tous les modèles
    """
    global models

    # Modèles à exclure de la sauvegarde par défaut
    if exclude_from_save is None:
        exclude_from_save = ['DecisionTree', 'RandomForest']

    models = {
        'SVC': SVC(),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'Gaussian': GaussianNB(),
        'LinearSVC': LinearSVC(),
        'SGD': SGDClassifier(),
        'DecisionTree': DecisionTreeClassifier(),
        'RandomForest': RandomForestClassifier(n_estimators=100),
        'XGBoost': XGBClassifier(learning_rate=0.0001)
    }

    results = {}
    best_accuracy = 0
    best_model = None
    best_model_name = None

    for name, model in models.items():
        # Entraînement du modèle
        model.fit(X_train, y_train)

        # Prédictions
        y_pred = model.predict(X_test)

        # Calcul du score
        accuracy = round(model.score(X_train, y_train) * 100, 2)
        results[name] = accuracy

        # Garder trace du meilleur modèle (seulement parmi ceux autorisés à être sauvegardés)
        if name not in exclude_from_save and accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model = model
            best_model_name = name

    # Sauvegarder le meilleur modèle (seulement si ce n'est pas dans la liste d'exclusion)
    if best_model is not None:
        with open(f'{df_type}_best_model_{best_model_name}.pkl', 'wb') as f:
            pickle.dump(best_model, f)
        print(f"Meilleur modèle - {df_type}({best_model_name}) sauvegardé avec une précision de {best_accuracy}%")

        # Afficher les modèles exclus s'ils étaient les meilleurs
        excluded_best = []
        for name, accuracy in results.items():
            if name in exclude_from_save and accuracy > best_accuracy:
                excluded_best.append((name, accuracy))

        if excluded_best:
            print(f"Note: Modèles exclus de la sauvegarde mais avec de meilleures performances:")
            for name, acc in excluded_best:
                print(f"  - {name}: {acc}%")
        print("\n==========================================\n")
    else:
        print(f"Aucun modèle valide trouvé pour la sauvegarde (tous exclus)")

    return results

In [109]:
def explore_columns(train_test_results):
    """Fonction pour explorer toutes les colonnes des ensembles X"""
    print("\n=== EXPLORATION DES COLONNES ===")

    for key, data in train_test_results.items():
        if key.startswith('X_') and not key.startswith('X_train') and not key.startswith('X_test'):
            print(f"\n{key}:")
            print(f"  Shape: {data.shape}")
            print(f"  Colonnes ({len(data.columns)}):")
            for i, col in enumerate(data.columns, 1):
                print(f"    {i:2d}. {col}")

# MPI TRAIN AND STORE MODELS

In [110]:
doc1_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc1/doc1_df_train.csv");
doc2_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc2/doc2_df_train.csv");
doc3_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc3/doc3_df_train.csv");

In [111]:
train_dfs = [doc1_df_train, doc2_df_train, doc3_df_train]

dataframes_prececed = []

for i in range(len(train_dfs)):
    train_df = train_dfs[i]
    df_admi_non_admi, df_session, df_mention = process_dataframes_flexible(train_df)
    dataframes_prececed.append(dict(doc="doc" + str(i + 1),
                                    admi_non_admi=df_admi_non_admi,
                                    session=df_session,
                                    mention=df_mention))

In [112]:
df_admi_non_admi = dataframes_prececed[0].get('admi_non_admi')
df_session = dataframes_prececed[0].get('session')
df_mention = dataframes_prececed[0].get('mention')

config = {
  'admin_non_admin': {
      'df': df_admi_non_admi,
      'target': 'RESULTAT',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'session': {
      'df': df_session,
      'target': 'SESSION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'mention': {
      'df': df_mention,
      'target': 'MENTION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  }
}

splited_for_doc1 = split_train_test_flexible(config)


X_train_admin_non_admin = splited_for_doc1.get('X_train_admin_non_admin')
y_train_admin_non_admin = splited_for_doc1.get('y_train_admin_non_admin')
X_test_admin_non_admin = splited_for_doc1.get('X_test_admin_non_admin')

X_train_session = splited_for_doc1.get('X_train_session')
y_train_session = splited_for_doc1.get('y_train_session')
X_test_session = splited_for_doc1.get('X_test_session')

X_train_mention = splited_for_doc1.get('X_train_mention')
y_train_mention = splited_for_doc1.get('y_train_mention')
X_test_mention = splited_for_doc1.get('X_test_mention')


# scores_admi_non_admin = train_evaluate_models(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin)
# scores_session = train_evaluate_models(X_train_session, y_train_session, X_test_session)
# scores_mention = train_evaluate_models(X_train_mention, y_train_mention, X_test_mention)

print(X_train_session.columns)
print(X_train_admin_non_admin.columns)
print(X_train_mention.columns)

print("=================== DOC 1 =========================")

scores_admi_non_admin = train_evaluate_models_flexible(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin, "admi_non_admi")
scores_session = train_evaluate_models_flexible(X_train_session, y_train_session, X_test_session, "session")
scores_mention = train_evaluate_models_flexible(X_train_mention, y_train_mention, X_test_mention, "mention")



# Pour afficher les résultats:
print("================Admi Non Admi========================")
for model, accuracy in scores_admi_non_admin.items():
    print(f'{model}: {accuracy}%')
print("================Admi Non Admi========================")
print("\n")

print("================ Session ========================")
# Pour afficher les résultats:
for model, accuracy in scores_session.items():
    print(f'{model}: {accuracy}%')
print("================ Session ========================")
print("\n")

print("================ Mention ========================")
# Pour afficher les résultats:
for model, accuracy in scores_mention.items():
    print(f'{model}: {accuracy}%')
print("================ Mention ========================")
print("\n")

Index(['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
       'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
       'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
       'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
       'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
       'Ets. de provenance_Encode', 'Centre d'Ec._Encode',
       'Académie de l'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode'],
      dtype='object')
Index(['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
       'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
       'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
       'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
       'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
       'Ets. de provenance_Encode', 'Centre d'Ec._Encode',
       'Académie de l'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode'],
      dtype='objec

In [113]:
df_admi_non_admi = dataframes_prececed[1].get('admi_non_admi')
df_session = dataframes_prececed[1].get('session')
df_mention = dataframes_prececed[1].get('mention')

config = {
  'admin_non_admin': {
      'df': df_admi_non_admi,
      'target': 'RESULTAT',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'session': {
      'df': df_session,
      'target': 'SESSION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'mention': {
      'df': df_mention,
      'target': 'MENTION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  }
}

splited_for_doc2 = split_train_test_flexible(config)


X_train_admin_non_admin = splited_for_doc2.get('X_train_admin_non_admin')
y_train_admin_non_admin = splited_for_doc2.get('y_train_admin_non_admin')
X_test_admin_non_admin = splited_for_doc2.get('X_test_admin_non_admin')

X_train_session = splited_for_doc2.get('X_train_session')
y_train_session = splited_for_doc2.get('y_train_session')
X_test_session = splited_for_doc2.get('X_test_session')

X_train_mention = splited_for_doc2.get('X_train_mention')
y_train_mention = splited_for_doc2.get('y_train_mention')
X_test_mention = splited_for_doc2.get('X_test_mention')


# scores_admi_non_admin = train_evaluate_models(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin)
# scores_session = train_evaluate_models(X_train_session, y_train_session, X_test_session)
# scores_mention = train_evaluate_models(X_train_mention, y_train_mention, X_test_mention)

print(X_train_session.columns)
print(X_train_admin_non_admin.columns)
print(X_train_mention.columns)

print("=================== DOC 2 =========================")

scores_admi_non_admin = train_evaluate_models_flexible(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin, "admi_non_admi")
scores_session = train_evaluate_models_flexible(X_train_session, y_train_session, X_test_session, "session")
scores_mention = train_evaluate_models_flexible(X_train_mention, y_train_mention, X_test_mention, "mention")


# Pour afficher les résultats:
print("================Admi Non Admi========================")
for model, accuracy in scores_admi_non_admin.items():
    print(f'{model}: {accuracy}%')
print("================Admi Non Admi========================")
print("\n")

print("================ Session ========================")
# Pour afficher les résultats:
for model, accuracy in scores_session.items():
    print(f'{model}: {accuracy}%')
print("================ Session ========================")
print("\n")

print("================ Mention ========================")
# Pour afficher les résultats:
for model, accuracy in scores_mention.items():
    print(f'{model}: {accuracy}%')
print("================ Mention ========================")
print("\n")

Index(['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
       'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
       'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
       'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
       'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
       'Ets. de provenance_Encode', 'Centre d'Ec._Encode',
       'Académie de l'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode'],
      dtype='object')
Index(['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
       'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
       'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
       'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
       'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
       'Ets. de provenance_Encode', 'Centre d'Ec._Encode',
       'Académie de l'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode'],
      dtype='objec

In [114]:
df_admi_non_admi = dataframes_prececed[2].get('admi_non_admi')
df_session = dataframes_prececed[2].get('session')
df_mention = dataframes_prececed[2].get('mention')

config = {
  'admin_non_admin': {
      'df': df_admi_non_admi,
      'target': 'RESULTAT',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'session': {
      'df': df_session,
      'target': 'SESSION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'mention': {
      'df': df_mention,
      'target': 'MENTION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  }
}

splited_for_doc3 = split_train_test_flexible(config)


X_train_admin_non_admin = splited_for_doc3.get('X_train_admin_non_admin')
y_train_admin_non_admin = splited_for_doc3.get('y_train_admin_non_admin')
X_test_admin_non_admin = splited_for_doc3.get('X_test_admin_non_admin')

X_train_session = splited_for_doc3.get('X_train_session')
y_train_session = splited_for_doc3.get('y_train_session')
X_test_session = splited_for_doc3.get('X_test_session')

X_train_mention = splited_for_doc3.get('X_train_mention')
y_train_mention = splited_for_doc3.get('y_train_mention')
X_test_mention = splited_for_doc3.get('X_test_mention')


# scores_admi_non_admin = train_evaluate_models(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin)
# scores_session = train_evaluate_models(X_train_session, y_train_session, X_test_session)
# scores_mention = train_evaluate_models(X_train_mention, y_train_mention, X_test_mention)

print(X_train_session.columns)
print(X_train_admin_non_admin.columns)
print(X_train_mention.columns)

print("=================== DOC 3 =========================")

scores_admi_non_admin = train_evaluate_models_flexible(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin, "admi_non_admi")
scores_session = train_evaluate_models_flexible(X_train_session, y_train_session, X_test_session, "session")
scores_mention = train_evaluate_models_flexible(X_train_mention, y_train_mention, X_test_mention, "mention")


# Pour afficher les résultats:
print("================Admi Non Admi========================")
for model, accuracy in scores_admi_non_admin.items():
    print(f'{model}: {accuracy}%')
print("================Admi Non Admi========================")
print("\n")

print("================ Session ========================")
# Pour afficher les résultats:
for model, accuracy in scores_session.items():
    print(f'{model}: {accuracy}%')
print("================ Session ========================")
print("\n")

print("================ Mention ========================")
# Pour afficher les résultats:
for model, accuracy in scores_mention.items():
    print(f'{model}: {accuracy}%')
print("================ Mention ========================")
print("\n")

Index(['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
       'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
       'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
       'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
       'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
       'Ets. de provenance_Encode', 'Centre d'Ec._Encode',
       'Académie de l'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode'],
      dtype='object')
Index(['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
       'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
       'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
       'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
       'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
       'Ets. de provenance_Encode', 'Centre d'Ec._Encode',
       'Académie de l'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode'],
      dtype='objec

# PCSM TRAIN AND STORE MODELS

In [115]:
doc1_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc1/doc1_df_train.csv");
doc2_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc2/doc2_df_train.csv");
doc3_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc3/doc3_df_train.csv");

In [116]:
train_dfs = [doc1_df_train, doc2_df_train, doc3_df_train]

dataframes_prececed = []

for i in range(len(train_dfs)):
    train_df = train_dfs[i]
    df_admi_non_admi, df_session, df_mention = process_dataframes_flexible(train_df)
    dataframes_prececed.append(dict(doc="doc" + str(i + 1),
                                    admi_non_admi=df_admi_non_admi,
                                    session=df_session,
                                    mention=df_mention))

In [117]:
df_admi_non_admi = dataframes_prececed[0].get('admi_non_admi')
df_session = dataframes_prececed[0].get('session')
df_mention = dataframes_prececed[0].get('mention')

config = {
  'admin_non_admin': {
      'df': df_admi_non_admi,
      'target': 'RESULTAT',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'session': {
      'df': df_session,
      'target': 'SESSION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'mention': {
      'df': df_mention,
      'target': 'MENTION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  }
}

splited_for_doc1 = split_train_test_flexible(config)


X_train_admin_non_admin = splited_for_doc1.get('X_train_admin_non_admin')
y_train_admin_non_admin = splited_for_doc1.get('y_train_admin_non_admin')
X_test_admin_non_admin = splited_for_doc1.get('X_test_admin_non_admin')

X_train_session = splited_for_doc1.get('X_train_session')
y_train_session = splited_for_doc1.get('y_train_session')
X_test_session = splited_for_doc1.get('X_test_session')

X_train_mention = splited_for_doc1.get('X_train_mention')
y_train_mention = splited_for_doc1.get('y_train_mention')
X_test_mention = splited_for_doc1.get('X_test_mention')


# scores_admi_non_admin = train_evaluate_models(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin)
# scores_session = train_evaluate_models(X_train_session, y_train_session, X_test_session)
# scores_mention = train_evaluate_models(X_train_mention, y_train_mention, X_test_mention)

print("=================== DOC 1 =========================")

scores_admi_non_admin = train_evaluate_models_flexible(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin, "admi_non_admi")
scores_session = train_evaluate_models_flexible(X_train_session, y_train_session, X_test_session, "session")
scores_mention = train_evaluate_models_flexible(X_train_mention, y_train_mention, X_test_mention, "mention")



# Pour afficher les résultats:
print("================Admi Non Admi========================")
for model, accuracy in scores_admi_non_admin.items():
    print(f'{model}: {accuracy}%')
print("================Admi Non Admi========================")
print("\n")

print("================ Session ========================")
# Pour afficher les résultats:
for model, accuracy in scores_session.items():
    print(f'{model}: {accuracy}%')
print("================ Session ========================")
print("\n")

print("================ Mention ========================")
# Pour afficher les résultats:
for model, accuracy in scores_mention.items():
    print(f'{model}: {accuracy}%')
print("================ Mention ========================")
print("\n")

=================== DOC 1 =========================
Meilleur modèle - admi_non_admi(KNN) sauvegardé avec une précision de 74.26%
Note: Modèles exclus de la sauvegarde mais avec de meilleures performances:
  - DecisionTree: 100.0%
  - RandomForest: 100.0%


Meilleur modèle - session(KNN) sauvegardé avec une précision de 81.27%
Note: Modèles exclus de la sauvegarde mais avec de meilleures performances:
  - DecisionTree: 100.0%
  - RandomForest: 100.0%


Meilleur modèle - mention(LinearSVC) sauvegardé avec une précision de 88.71%
Note: Modèles exclus de la sauvegarde mais avec de meilleures performances:
  - DecisionTree: 100.0%
  - RandomForest: 100.0%


================Admi Non Admi========================
SVC: 59.08%
KNN: 74.26%
Gaussian: 60.12%
LinearSVC: 70.09%
SGD: 60.57%
DecisionTree: 100.0%
RandomForest: 100.0%
XGBoost: 59.08%
================Admi Non Admi========================


================ Session ========================
SVC: 77.53%
KNN: 81.27%
Gaussian: 78.65%
LinearSVC

In [118]:
df_admi_non_admi = dataframes_prececed[1].get('admi_non_admi')
df_session = dataframes_prececed[1].get('session')
df_mention = dataframes_prececed[1].get('mention')

config = {
  'admin_non_admin': {
      'df': df_admi_non_admi,
      'target': 'RESULTAT',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'session': {
      'df': df_session,
      'target': 'SESSION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'mention': {
      'df': df_mention,
      'target': 'MENTION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  }
}

splited_for_doc2 = split_train_test_flexible(config)


X_train_admin_non_admin = splited_for_doc2.get('X_train_admin_non_admin')
y_train_admin_non_admin = splited_for_doc2.get('y_train_admin_non_admin')
X_test_admin_non_admin = splited_for_doc2.get('X_test_admin_non_admin')

X_train_session = splited_for_doc2.get('X_train_session')
y_train_session = splited_for_doc2.get('y_train_session')
X_test_session = splited_for_doc2.get('X_test_session')

X_train_mention = splited_for_doc2.get('X_train_mention')
y_train_mention = splited_for_doc2.get('y_train_mention')
X_test_mention = splited_for_doc2.get('X_test_mention')


# scores_admi_non_admin = train_evaluate_models(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin)
# scores_session = train_evaluate_models(X_train_session, y_train_session, X_test_session)
# scores_mention = train_evaluate_models(X_train_mention, y_train_mention, X_test_mention)

print("=================== DOC 2 =========================")

scores_admi_non_admin = train_evaluate_models_flexible(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin, "admi_non_admi")
scores_session = train_evaluate_models_flexible(X_train_session, y_train_session, X_test_session, "session")
scores_mention = train_evaluate_models_flexible(X_train_mention, y_train_mention, X_test_mention, "mention")


# Pour afficher les résultats:
print("================Admi Non Admi========================")
for model, accuracy in scores_admi_non_admin.items():
    print(f'{model}: {accuracy}%')
print("================Admi Non Admi========================")
print("\n")

print("================ Session ========================")
# Pour afficher les résultats:
for model, accuracy in scores_session.items():
    print(f'{model}: {accuracy}%')
print("================ Session ========================")
print("\n")

print("================ Mention ========================")
# Pour afficher les résultats:
for model, accuracy in scores_mention.items():
    print(f'{model}: {accuracy}%')
print("================ Mention ========================")
print("\n")

=================== DOC 2 =========================
Meilleur modèle - admi_non_admi(KNN) sauvegardé avec une précision de 75.0%
Note: Modèles exclus de la sauvegarde mais avec de meilleures performances:
  - DecisionTree: 100.0%
  - RandomForest: 100.0%


Meilleur modèle - session(LinearSVC) sauvegardé avec une précision de 84.53%
Note: Modèles exclus de la sauvegarde mais avec de meilleures performances:
  - DecisionTree: 100.0%
  - RandomForest: 100.0%


Meilleur modèle - mention(LinearSVC) sauvegardé avec une précision de 93.33%
Note: Modèles exclus de la sauvegarde mais avec de meilleures performances:
  - DecisionTree: 100.0%
  - RandomForest: 100.0%


================Admi Non Admi========================
SVC: 58.48%
KNN: 75.0%
Gaussian: 59.38%
LinearSVC: 67.41%
SGD: 59.67%
DecisionTree: 100.0%
RandomForest: 100.0%
XGBoost: 58.48%
================Admi Non Admi========================


================ Session ========================
SVC: 79.14%
KNN: 82.73%
Gaussian: 80.22%
Linea

In [119]:
df_admi_non_admi = dataframes_prececed[2].get('admi_non_admi')
df_session = dataframes_prececed[2].get('session')
df_mention = dataframes_prececed[2].get('mention')

config = {
  'admin_non_admin': {
      'df': df_admi_non_admi,
      'target': 'RESULTAT',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'session': {
      'df': df_session,
      'target': 'SESSION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'mention': {
      'df': df_mention,
      'target': 'MENTION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  }
}

splited_for_doc3 = split_train_test_flexible(config)


X_train_admin_non_admin = splited_for_doc3.get('X_train_admin_non_admin')
y_train_admin_non_admin = splited_for_doc3.get('y_train_admin_non_admin')
X_test_admin_non_admin = splited_for_doc3.get('X_test_admin_non_admin')

X_train_session = splited_for_doc3.get('X_train_session')
y_train_session = splited_for_doc3.get('y_train_session')
X_test_session = splited_for_doc3.get('X_test_session')

X_train_mention = splited_for_doc3.get('X_train_mention')
y_train_mention = splited_for_doc3.get('y_train_mention')
X_test_mention = splited_for_doc3.get('X_test_mention')


# scores_admi_non_admin = train_evaluate_models(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin)
# scores_session = train_evaluate_models(X_train_session, y_train_session, X_test_session)
# scores_mention = train_evaluate_models(X_train_mention, y_train_mention, X_test_mention)

print("=================== DOC 3 =========================")

scores_admi_non_admin = train_evaluate_models_flexible(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin, "admi_non_admi")
scores_session = train_evaluate_models_flexible(X_train_session, y_train_session, X_test_session, "session")
scores_mention = train_evaluate_models_flexible(X_train_mention, y_train_mention, X_test_mention, "mention")


# Pour afficher les résultats:
print("================Admi Non Admi========================")
for model, accuracy in scores_admi_non_admin.items():
    print(f'{model}: {accuracy}%')
print("================Admi Non Admi========================")
print("\n")

print("================ Session ========================")
# Pour afficher les résultats:
for model, accuracy in scores_session.items():
    print(f'{model}: {accuracy}%')
print("================ Session ========================")
print("\n")

print("================ Mention ========================")
# Pour afficher les résultats:
for model, accuracy in scores_mention.items():
    print(f'{model}: {accuracy}%')
print("================ Mention ========================")
print("\n")

=================== DOC 3 =========================
Meilleur modèle - admi_non_admi(KNN) sauvegardé avec une précision de 73.07%
Note: Modèles exclus de la sauvegarde mais avec de meilleures performances:
  - DecisionTree: 100.0%
  - RandomForest: 100.0%


Meilleur modèle - session(KNN) sauvegardé avec une précision de 81.04%
Note: Modèles exclus de la sauvegarde mais avec de meilleures performances:
  - DecisionTree: 100.0%
  - RandomForest: 100.0%


Meilleur modèle - mention(LinearSVC) sauvegardé avec une précision de 85.71%
Note: Modèles exclus de la sauvegarde mais avec de meilleures performances:
  - DecisionTree: 100.0%
  - RandomForest: 100.0%


================Admi Non Admi========================
SVC: 61.31%
KNN: 73.07%
Gaussian: 61.76%
LinearSVC: 69.79%
SGD: 38.69%
DecisionTree: 100.0%
RandomForest: 100.0%
XGBoost: 61.31%
================Admi Non Admi========================


================ Session ========================
SVC: 78.44%
KNN: 81.04%
Gaussian: 33.83%
LinearSVC

# BCGS TRAIN AND STORE MODELS

In [120]:
doc1_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc1/doc1_df_train.csv");
doc2_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc2/doc2_df_train.csv");
doc3_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc3/doc3_df_train.csv");

In [121]:
train_dfs = [doc1_df_train, doc2_df_train, doc3_df_train]

dataframes_prececed = []

for i in range(len(train_dfs)):
    train_df = train_dfs[i]
    df_admi_non_admi, df_session, df_mention = process_dataframes_flexible(train_df)
    dataframes_prececed.append(dict(doc="doc" + str(i + 1),
                                    admi_non_admi=df_admi_non_admi,
                                    session=df_session,
                                    mention=df_mention))

In [122]:
df_admi_non_admi = dataframes_prececed[0].get('admi_non_admi')
df_session = dataframes_prececed[0].get('session')
df_mention = dataframes_prececed[0].get('mention')

config = {
  'admin_non_admin': {
      'df': df_admi_non_admi,
      'target': 'RESULTAT',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'session': {
      'df': df_session,
      'target': 'SESSION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'mention': {
      'df': df_mention,
      'target': 'MENTION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  }
}

splited_for_doc1 = split_train_test_flexible(config)


X_train_admin_non_admin = splited_for_doc1.get('X_train_admin_non_admin')
y_train_admin_non_admin = splited_for_doc1.get('y_train_admin_non_admin')
X_test_admin_non_admin = splited_for_doc1.get('X_test_admin_non_admin')

X_train_session = splited_for_doc1.get('X_train_session')
y_train_session = splited_for_doc1.get('y_train_session')
X_test_session = splited_for_doc1.get('X_test_session')

X_train_mention = splited_for_doc1.get('X_train_mention')
y_train_mention = splited_for_doc1.get('y_train_mention')
X_test_mention = splited_for_doc1.get('X_test_mention')


# scores_admi_non_admin = train_evaluate_models(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin)
# scores_session = train_evaluate_models(X_train_session, y_train_session, X_test_session)
# scores_mention = train_evaluate_models(X_train_mention, y_train_mention, X_test_mention)


print(X_train_session.columns)
print(X_train_admin_non_admin.columns)
print(X_train_mention.columns)

print("=================== DOC 1 =========================")

scores_admi_non_admin = train_evaluate_models_flexible(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin, "admi_non_admi")
scores_session = train_evaluate_models_flexible(X_train_session, y_train_session, X_test_session, "session")
scores_mention = train_evaluate_models_flexible(X_train_mention, y_train_mention, X_test_mention, "mention")



# Pour afficher les résultats:
print("================Admi Non Admi========================")
for model, accuracy in scores_admi_non_admin.items():
    print(f'{model}: {accuracy}%')
print("================Admi Non Admi========================")
print("\n")

print("================ Session ========================")
# Pour afficher les résultats:
for model, accuracy in scores_session.items():
    print(f'{model}: {accuracy}%')
print("================ Session ========================")
print("\n")

print("================ Mention ========================")
# Pour afficher les résultats:
for model, accuracy in scores_mention.items():
    print(f'{model}: {accuracy}%')
print("================ Mention ========================")
print("\n")

Index(['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
       'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
       'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
       'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
       'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
       'Ets. de provenance_Encode', 'Centre d'Ec._Encode',
       'Académie de l'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode'],
      dtype='object')
Index(['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
       'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
       'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
       'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
       'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
       'Ets. de provenance_Encode', 'Centre d'Ec._Encode',
       'Académie de l'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode'],
      dtype='objec

In [123]:
df_admi_non_admi = dataframes_prececed[1].get('admi_non_admi')
df_session = dataframes_prececed[1].get('session')
df_mention = dataframes_prececed[1].get('mention')

config = {
  'admin_non_admin': {
      'df': df_admi_non_admi,
      'target': 'RESULTAT',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'session': {
      'df': df_session,
      'target': 'SESSION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'mention': {
      'df': df_mention,
      'target': 'MENTION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  }
}

splited_for_doc2 = split_train_test_flexible(config)


X_train_admin_non_admin = splited_for_doc2.get('X_train_admin_non_admin')
y_train_admin_non_admin = splited_for_doc2.get('y_train_admin_non_admin')
X_test_admin_non_admin = splited_for_doc2.get('X_test_admin_non_admin')

X_train_session = splited_for_doc2.get('X_train_session')
y_train_session = splited_for_doc2.get('y_train_session')
X_test_session = splited_for_doc2.get('X_test_session')

X_train_mention = splited_for_doc2.get('X_train_mention')
y_train_mention = splited_for_doc2.get('y_train_mention')
X_test_mention = splited_for_doc2.get('X_test_mention')


# scores_admi_non_admin = train_evaluate_models(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin)
# scores_session = train_evaluate_models(X_train_session, y_train_session, X_test_session)
# scores_mention = train_evaluate_models(X_train_mention, y_train_mention, X_test_mention)

print(X_train_session.columns)
print(X_train_admin_non_admin.columns)
print(X_train_mention.columns)

print("=================== DOC 2 =========================")

scores_admi_non_admin = train_evaluate_models_flexible(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin, "admi_non_admi")
scores_session = train_evaluate_models_flexible(X_train_session, y_train_session, X_test_session, "session")
scores_mention = train_evaluate_models_flexible(X_train_mention, y_train_mention, X_test_mention, "mention")


# Pour afficher les résultats:
print("================Admi Non Admi========================")
for model, accuracy in scores_admi_non_admin.items():
    print(f'{model}: {accuracy}%')
print("================Admi Non Admi========================")
print("\n")

print("================ Session ========================")
# Pour afficher les résultats:
for model, accuracy in scores_session.items():
    print(f'{model}: {accuracy}%')
print("================ Session ========================")
print("\n")

print("================ Mention ========================")
# Pour afficher les résultats:
for model, accuracy in scores_mention.items():
    print(f'{model}: {accuracy}%')
print("================ Mention ========================")
print("\n")

Index(['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
       'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
       'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
       'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
       'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
       'Ets. de provenance_Encode', 'Centre d'Ec._Encode',
       'Académie de l'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode'],
      dtype='object')
Index(['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
       'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
       'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
       'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
       'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
       'Ets. de provenance_Encode', 'Centre d'Ec._Encode',
       'Académie de l'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode'],
      dtype='objec

In [126]:
df_admi_non_admi = dataframes_prececed[2].get('admi_non_admi')
df_session = dataframes_prececed[2].get('session')
df_mention = dataframes_prececed[2].get('mention')

config = {
  'admin_non_admin': {
      'df': df_admi_non_admi,
      'target': 'RESULTAT',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'session': {
      'df': df_session,
      'target': 'SESSION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  },
  'mention': {
      'df': df_mention,
      'target': 'MENTION',
      'drop_columns': ['Rang L1', 'Score L1', 'RESULTAT APP EVALUATION', 'NIVEAU', 'Année de l\'Extrait EC',
                        'Sexe','Série','RESULTAT_Encode','SESSION_Encode','MENTION_Encode','Residence perf.','Résidence','Ets. de provenance',
                        'Centre d\'Ec.','Académie de l\'Ets. Prov.','REGION_DE_NAISSANCE','Femme','Homme','S1','S2','S3']
  }
}

splited_for_doc3 = split_train_test_flexible(config)


X_train_admin_non_admin = splited_for_doc3.get('X_train_admin_non_admin')
y_train_admin_non_admin = splited_for_doc3.get('y_train_admin_non_admin')
X_test_admin_non_admin = splited_for_doc3.get('X_test_admin_non_admin')

X_train_session = splited_for_doc3.get('X_train_session')
y_train_session = splited_for_doc3.get('y_train_session')
X_test_session = splited_for_doc3.get('X_test_session')

X_train_mention = splited_for_doc3.get('X_train_mention')
y_train_mention = splited_for_doc3.get('y_train_mention')
X_test_mention = splited_for_doc3.get('X_test_mention')


# scores_admi_non_admin = train_evaluate_models(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin)
# scores_session = train_evaluate_models(X_train_session, y_train_session, X_test_session)
# scores_mention = train_evaluate_models(X_train_mention, y_train_mention, X_test_mention)

print(X_train_session.columns)
print(X_train_admin_non_admin.columns)
print(X_train_mention.columns)

print(y_train_mention.name)
print(y_train_session.name)
print(y_train_admin_non_admin.name)


print(y_train_mention)
print(y_train_session)
print(y_train_admin_non_admin)

print("=================== DOC 3 =========================")

scores_admi_non_admin = train_evaluate_models_flexible(X_train_admin_non_admin, y_train_admin_non_admin, X_test_admin_non_admin, "admi_non_admi")
scores_session = train_evaluate_models_flexible(X_train_session, y_train_session, X_test_session, "session")
scores_mention = train_evaluate_models_flexible(X_train_mention, y_train_mention, X_test_mention, "mention")


# Pour afficher les résultats:
print("================Admi Non Admi========================")
for model, accuracy in scores_admi_non_admin.items():
    print(f'{model}: {accuracy}%')
print("================Admi Non Admi========================")
print("\n")

print("================ Session ========================")
# Pour afficher les résultats:
for model, accuracy in scores_session.items():
    print(f'{model}: {accuracy}%')
print("================ Session ========================")
print("\n")

print("================ Mention ========================")
# Pour afficher les résultats:
for model, accuracy in scores_mention.items():
    print(f'{model}: {accuracy}%')
print("================ Mention ========================")
print("\n")

Index(['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
       'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
       'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
       'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
       'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
       'Ets. de provenance_Encode', 'Centre d'Ec._Encode',
       'Académie de l'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode'],
      dtype='object')
Index(['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
       'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
       'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
       'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
       'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
       'Ets. de provenance_Encode', 'Centre d'Ec._Encode',
       'Académie de l'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode'],
      dtype='objec